# 정규표현식 기초: 검색과 검증을 구분하기

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 문자열·불리언·함수. 표준 라이브러리 re

## 오늘 설명할 수 있어야 하는 것

- 문자 패턴과 전체 문자열의 유효성 검증을 구분한다.
- 맞는 예와 비슷하지만 틀린 예를 함께 제시해 AI의 패턴을 검토한다.


## 1. 먼저 원리를 읽기

정규표현식은 반복되는 문자열 규칙을 표현합니다. 문법을 전부 암기하기보다 **무엇을 허용하고 무엇을 거절할지** 예로 설명하세요.
`search`는 부분 문자열을 찾고, `fullmatch`는 전체 문자열이 규칙을 만족하는지 확인합니다. 여기서는 합성 지점 ID `ST-`와 ASCII 숫자 3개만 허용합니다. 숫자처럼 보이는 모든 문자를 허용하는 계약은 아닙니다.


## 2. 실행 전 예측하기

`search`와 `fullmatch`는 `"prefix ST-012 suffix"`에 각각 무엇을 반환할까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
import re

pattern = re.compile(r"ST-[0-9]{3}")
for text in ["ST-012", "prefix ST-012 suffix", "ST-12", "ST-0123"]:
    found = pattern.search(text)
    valid = pattern.fullmatch(text)
    print(repr(text), "검색:", bool(found), "전체 검증:", bool(valid))
assert pattern.search("prefix ST-012 suffix") is not None
assert pattern.fullmatch("prefix ST-012 suffix") is None


### 관찰을 원리와 연결하기

부분적으로 맞는 문자열을 찾는 것과 입력 전체가 올바른 것은 다릅니다. AI에게 “패턴을 만들어 줘”만 말하지 말고 추출·검증·치환 중 목적을 정하세요.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
text = "A ST-012, B ST-003"  # 합성 지점 ID
masked = re.sub(r"ST-[0-9]{3}", "ST-***", text)
print(masked)
assert masked == "A ST-***, B ST-***"
# 이 단순 치환은 ST-0123의 일부도 바꿀 수 있습니다. 경계 정책이 필요합니다.
print("경계 미지정:", re.sub(r"ST-[0-9]{3}", "ST-***", "ST-0123"))


## 3. 작은 기능을 직접 완성하기

`is_station_id(text)`는 문자열 전체가 `ST-` + ASCII 숫자 정확히 3개일 때 `True`, 아니면 `False`를 반환합니다.
앞뒤 공백, 소문자 접두사, 자릿수 부족·초과, 전각 숫자는 허용하지 않습니다. 입력은 문자열입니다.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def is_station_id(text):
    # TODO: 검색이 필요한지 전체 검증이 필요한지 먼저 정하세요.
    raise NotImplementedError("허용·거절 예를 구현하세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    cases = {"ST-000": True, "ST-012": True, "ST-999": True,
             "ST-12": False, "ST-0123": False, "xST-012": False,
             "ST-012 ": False, "ST-012\n": False, "st-012": False,
             "ST-１２３": False, "": False}
    for text, expected in cases.items():
        result = candidate(text)
        assert isinstance(result, bool) and result == expected, (text, result, expected)
    print("공개 점검 통과: 정상·자릿수·공백·경계·ASCII")

try:
    check_answer(is_station_id)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
def ai_suggestion(text):
    return re.search(r"ST-\d{3}", text) is not None

for text in ["ST-0123", "xST-012", "ST-１２３"]:
    print(repr(text), ai_suggestion(text))
    assert ai_suggestion(text) is True


### 검토 질문

부분 검색과 Unicode 숫자 허용 때문에 계약보다 넓은 입력이 통과합니다. 정상 예만 늘리지 말고 “비슷하지만 거절해야 하는 입력”을 시험 목록에 넣으세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

지점 ID의 허용 문자·대소문자·자릿수·공백 정책을 정하고, 추출이 아닌 전체 검증과 bool 반환을 요구한다. 정상 2개와 거절 반례 4개를 직접 작성해 전달한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

문장 안의 지점 ID를 추출하는 작업으로 바뀌면 공백·문장부호·영문자 옆의 ID를 어떻게 다룰지 먼저 정의하세요. 같은 패턴을 그대로 써도 되는지 설명하세요.


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
